# Stage 1 — Load & Explore Protein2Text-QA

This notebook covers loading, cleaning, and fine-tuning protein encoders, followed by CLIP-style alignment with text.

## 1 · Install dependencies

In [1]:
# Run once — comment out after the first execution
%pip install -q datasets pandas pyarrow tqdm transformers sentencepiece torch peft scikit-learn seaborn matplotlib

## 2 · Imports & configuration

In [2]:
import re
import random
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_NAME     = "tumorailab/Protein2Text-QA"
TRUNCATE_SEQ_LEN = 256   # truncate (not filter) every aa sequence to this length
N_TRAIN          = 100   # number of rows to keep for training
N_TEST           = 100   # number of rows to keep for testing
RANDOM_SEED      = 42
OUTPUT_DIR       = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Truncate sequences to : {TRUNCATE_SEQ_LEN} aa")
print(f"Train samples         : {N_TRAIN}")
print(f"Test  samples         : {N_TEST}")
print(f"Saving clean pairs to : {OUTPUT_DIR.resolve()}")

## 3 · Load the dataset

In [3]:
print(f"Loading '{DATASET_NAME}' from Hugging Face …")
ds = load_dataset(DATASET_NAME)
print(ds)

## 4 · Inspect schema & splits

In [4]:
print("=" * 60)
print("Available splits:", list(ds.keys()))
print()
for split_name, split_data in ds.items():
    print(f"  {split_name:20s}  rows={len(split_data):>8,}  features={list(split_data.features.keys())}")

In [5]:
# Pick whichever split is available as the primary reference
PRIMARY_SPLIT = "train" if "train" in ds else list(ds.keys())[0]
print(f"Primary split  : {PRIMARY_SPLIT}")
print("Column dtypes:")
print(ds[PRIMARY_SPLIT].features)

## 5 · Sample rows

In [6]:
# Show a few raw rows from the primary split
for i in range(3):
    print(f"\n── Sample {i} ──")
    row = ds[PRIMARY_SPLIT][i]
    for k, v in row.items():
        val_str = str(v)
        print(f"  {k}: {val_str[:300]}{'…' if len(val_str) > 300 else ''}")

In [7]:
# Convert the primary split to a Pandas DataFrame for easy exploration
df_train_raw = ds[PRIMARY_SPLIT].to_pandas()
df_train_raw.head(3)

In [8]:
df_train_raw.info()

## 6 · Identify sequence & answer columns

In [9]:
COLS = df_train_raw.columns.tolist()
print("All columns:", COLS)

def _find_col(candidates, columns):
    """Return the first candidate column name found in `columns`."""
    for c in candidates:
        if c in columns:
            return c
    return None

SEQ_COL = _find_col(
    ["amino_seq", "sequence", "protein_sequence", "seq", "aa_sequence", "protein"], COLS
)
ANS_COL = _find_col(
    ["conversations", "answer", "answers", "output", "response", "description", "text"], COLS
)
QST_COL = _find_col(
    ["question", "input", "query", "instruction"], COLS
)

# Flag whether the answer column is the conversations list format
IS_CONV = (ANS_COL == "conversations")

print(f"\nSequence column     : {SEQ_COL}")
print(f"Answer column       : {ANS_COL}")
print(f"Uses conversations? : {IS_CONV}")
print(f"Question column     : {QST_COL}")

if SEQ_COL is None or ANS_COL is None:
    raise ValueError(
        "Could not auto-detect sequence/answer columns. "
        f"Please set SEQ_COL and ANS_COL manually from: {COLS}"
    )

### 6a · Peek inside the `conversations` column

In [10]:
if IS_CONV:
    sample_conv = df_train_raw[ANS_COL].iloc[0]
    print("Type:", type(sample_conv))
    print("First conversation turns:")
    for turn in sample_conv:
        print(f"  from={turn.get('from','?'):10s} | value={str(turn.get('value',''))[:200]}")

## 7 · Explore raw sequence length distribution

In [11]:
df_train_raw["seq_len_raw"] = df_train_raw[SEQ_COL].str.len()

print("Raw sequence length statistics (before truncation):")
print(df_train_raw["seq_len_raw"].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

In [12]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3))
    df_train_raw["seq_len_raw"].clip(upper=1500).hist(bins=80, ax=ax)
    ax.axvline(TRUNCATE_SEQ_LEN, color="red", linestyle="--",
               label=f"truncation point ({TRUNCATE_SEQ_LEN} aa)")
    ax.set_xlabel("Sequence length (aa)")
    ax.set_ylabel("Count")
    ax.set_title(f"{PRIMARY_SPLIT} split — raw sequence length (clipped at 1 500)")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping histogram")

## 8 · Simplify AA sequences to 256

In [13]:
def first_sentence(text: str) -> str:
    """Return the first sentence of a string."""
    if not isinstance(text, str):
        return ""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return sentences[0].strip() if sentences else text.strip()

def extract_answer(cell) -> str:
    """Extract the first assistant reply from a conversations cell."""
    if not IS_CONV:
        return first_sentence(str(cell)) if cell is not None else ""
    if isinstance(cell, (list, pd.Series, tqdm)):
        for turn in cell:
            if isinstance(turn, dict) and turn.get("from", "").lower() in ("gpt", "assistant", "model"):
                return first_sentence(turn.get("value", ""))
    elif hasattr(cell, 'tolist'): # handle numpy array
        for turn in cell.tolist():
            if isinstance(turn, dict) and turn.get("from", "").lower() in ("gpt", "assistant", "model"):
                return first_sentence(turn.get("value", ""))
    return ""

def extract_question(cell) -> str:
    """Extract the first human question from a conversations cell."""
    if not IS_CONV:
        return ""
    turns = cell.tolist() if hasattr(cell, 'tolist') else cell
    if isinstance(turns, list):
        for turn in turns:
            if isinstance(turn, dict) and turn.get("from", "").lower() in ("human", "user"):
                val = turn.get("value", "")
                val = re.sub(r"<protein_sequence>\s*", "", val).strip()
                return val
    return ""

def clean_split(df: pd.DataFrame, n_samples: int) -> pd.DataFrame:
    """Clean, truncate, and sample a split DataFrame."""
    df = df.copy()
    df = df.dropna(subset=[SEQ_COL, ANS_COL])
    
    if IS_CONV:
        df["question"]    = df[ANS_COL].apply(extract_question)
        df["description"] = df[ANS_COL].apply(extract_answer)
    else:
        df["description"] = df[ANS_COL].apply(first_sentence)
    
    # TRUNCATE here
    df["sequence"] = df[SEQ_COL].astype(str).str[:TRUNCATE_SEQ_LEN]
    df["seq_len"]  = df["sequence"].str.len()
    
    df = df[df["description"].str.len() > 0]
    keep_cols = ["sequence", "description", "seq_len"]
    if "question" in df.columns: keep_cols.insert(1, "question")
    
    df = df[keep_cols].reset_index(drop=True)
    return df.sample(n=min(len(df), n_samples), random_state=RANDOM_SEED).reset_index(drop=True)

# Create the simplified datasets
df_all = ds[PRIMARY_SPLIT].to_pandas()
df_train_clean = clean_split(df_all.sample(frac=1, random_state=RANDOM_SEED).iloc[:len(df_all)//2], N_TRAIN)
df_test_clean  = clean_split(df_all.sample(frac=1, random_state=RANDOM_SEED).iloc[len(df_all)//2:], N_TEST)

print(f"Simplified Train rows: {len(df_train_clean)}")
print(f"Simplified Test rows:  {len(df_test_clean)}")

# Plot the new length hist
try:
    plt.figure(figsize=(8,3))
    df_train_clean["seq_len"].hist(bins=30, color='skyblue', edgecolor='black')
    plt.title(f"Truncated Sequence Lengths (max={TRUNCATE_SEQ_LEN})")
    plt.xlabel("Length (aa)")
    plt.ylabel("Count")
    plt.show()
except NameError:
    print("matplotlib/plt not defined")

## 9 · Save to disk

In [14]:
df_train_clean.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
df_test_clean.to_parquet(OUTPUT_DIR / "test.parquet", index=False)
print(f"Saved to {OUTPUT_DIR}")

# Protein Encoder FT

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ── Loading ───────────────────────────────────────────────────────────────────
MODEL_ID = "Rostlab/prot_t5_xl_uniref50"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model: {MODEL_ID} to {DEVICE} …")

tokenizer = T5Tokenizer.from_pretrained(MODEL_ID, do_lower_case=False)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID)
model = model.to(DEVICE)

if DEVICE == "cuda":
    model = model.half()

print("Model loaded successfully!")

## 10 · Fine-tuning with LoRA (MLM Task)

We randomly mask out ~15% of amino acids in the sequence and train the model to predict them.

In [23]:
import random
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader

# ── Handling Encoder-only vs Full Model ──
is_encoder_only = "Encoder" in type(model).__name__
pt_task_type    = None if is_encoder_only else TaskType.SEQ_2_SEQ_LM

print(f"Model type: {type(model).__name__} | Task Type: {pt_task_type}")

class T5EncoderWithMLMHead(nn.Module):
    """Wraps a T5EncoderModel with a linear head for MLM if needed."""
    def __init__(self, encoder, vocab_size):
        super().__init__()
        self.encoder = encoder
        self.lm_head = nn.Linear(encoder.config.d_model, vocab_size, bias=False)
        self.lm_head = self.lm_head.to(next(encoder.parameters()).dtype)
        
    def forward(self, input_ids, attention_mask, labels=None):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = enc_out.last_hidden_state.to(self.lm_head.weight.dtype)
        logits  = self.lm_head(hidden_states)
        loss    = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return type("MLMOut", (), {"loss": loss, "logits": logits})()

if is_encoder_only:
    trainable_model = T5EncoderWithMLMHead(model, tokenizer.vocab_size).to(DEVICE)
else:
    trainable_model = model

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05, bias="none", task_type=pt_task_type)
trainable_model = get_peft_model(trainable_model, lora_config)
trainable_model.print_trainable_parameters()

def mask_protein_mlm(sequence, mask_prob=0.15):
    aas    = list(re.sub(r"[UZOB]", "X", sequence.upper()))
    tokens = tokenizer(" ".join(aas), truncation=True, max_length=TRUNCATE_SEQ_LEN).input_ids
    labels = [-100] * len(tokens)
    for i in range(len(tokens)):
        if tokens[i] in tokenizer.all_special_ids: continue
        if random.random() < mask_prob:
            labels[i] = tokens[i]
            tokens[i] = tokenizer.convert_tokens_to_ids("X") if "X" in tokenizer.vocab else tokenizer.mask_token_id or 3
    return tokens, labels

class ProteinMLMDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return mask_protein_mlm(self.sequences[idx])

def collate_fn(batch):
    inps, tgts = zip(*batch)
    max_len = max(len(x) for x in inps)
    padded_inps = [x + [tokenizer.pad_token_id]*(max_len-len(x)) for x in inps]
    padded_tgts = [x + [-100]*(max_len-len(x)) for x in tgts]
    mask        = [[1]*len(x) + [0]*(max_len-len(x)) for x in inps]
    return torch.tensor(padded_inps).to(DEVICE), torch.tensor(mask).to(DEVICE), torch.tensor(padded_tgts).to(DEVICE)

train_loader = DataLoader(ProteinMLMDataset(df_train_clean["sequence"].tolist()), batch_size=4, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.AdamW(trainable_model.parameters(), lr=1e-4)
trainable_model.train()

print("\nStarting Training (1 Epoch) ...")
for step, (input_ids, attention_mask, labels) in enumerate(tqdm(train_loader, desc="Training")):
    outputs = trainable_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels.long())
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


## 11 · Embedding Visualization (t-SNE)

We extract embeddings for the test set, pool them, and project them into 2D.

In [25]:
import numpy as np
import seaborn as sns
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def get_pooled_embedding(sequence: str):
    """Extract sequence-level embedding (mean of token hidden states)."""
    seq_fmt = " ".join(list(re.sub(r"[UZOB]", "X", sequence.upper())))
    inputs  = tokenizer(seq_fmt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        target_model = model
        if hasattr(target_model, "encoder"):
            outputs = target_model.encoder(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
            emb = outputs.last_hidden_state
        else:
            outputs = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
            emb = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs.logits
    return emb.mean(dim=1).cpu().numpy().flatten()

print("Extracting embeddings for test set ...")
test_sequences = df_test_clean["sequence"].tolist()
embeddings = []
for seq in tqdm(test_sequences):
    embeddings.append(get_pooled_embedding(seq))
embeddings = np.array(embeddings)
print(f"Embeddings extracted: {embeddings.shape}") # [N_TEST, 1024]

def get_pseudo_label(desc):
    desc = desc.lower()
    if any(k in desc for k in ["membrane", "transmembrane"]): return "Membrane"
    if any(k in desc for k in ["enzyme", "catalytic", "hydrolase", "transferase"]): return "Enzyme"
    if any(k in desc for k in ["binding", "receptor"]): return "Binding/Receptor"
    if any(k in desc for k in ["secreted", "signal"]): return "Secreted"
    return "Other"
labels_pseudo = df_test_clean["description"].apply(get_pseudo_label)
tsne = TSNE(n_components=2, perplexity=15, random_state=42, init='pca', learning_rate='auto')
z = tsne.fit_transform(embeddings)
df_tsne = pd.DataFrame({"TSNE1": z[:, 0], "TSNE2": z[:, 1], "Label": labels_pseudo})
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_tsne, x="TSNE1", y="TSNE2", hue="Label", palette="viridis", s=100, alpha=0.8)
plt.title(f"t-SNE of Protein Embeddings (Fine-tuned ProtT5-XL)\nColored by Description Keywords")
plt.show()

# Stage 3 — CLIP Protein-Text Alignment

Align ProtT5 embeddings with MiniLM text embeddings using contrastive loss.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

TEXT_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
PROT_DIM, TEXT_DIM = 1024, 384
BATCH_SIZE, LR = 8, 1e-4

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
text_model     = AutoModel.from_pretrained(TEXT_MODEL_ID).to(DEVICE)

for p in model.parameters(): p.requires_grad = False
for p in text_model.parameters(): p.requires_grad = False

class ProteinTextCLIP(nn.Module):
    def __init__(self, prot_encoder, text_encoder, p_dim, t_dim):
        super().__init__()
        self.prot_encoder = prot_encoder
        self.text_encoder = text_encoder
        self.text_proj    = nn.Linear(t_dim, p_dim)
        self.logit_scale  = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
    def get_text_features(self, ids, mask):
        emb = self.text_encoder(ids, mask).last_hidden_state.mean(dim=1)
        return self.text_proj(emb.to(self.text_proj.weight.dtype))

    def get_prot_features(self, ids, mask):
        target = self.prot_encoder.encoder if hasattr(self.prot_encoder, 'encoder') else self.prot_encoder
        return target(ids, mask).last_hidden_state.mean(dim=1)

    def forward(self, p_ids, p_mask, t_ids, t_mask):
        pf = F.normalize(self.get_prot_features(p_ids, p_mask), p=2, dim=-1)
        tf = F.normalize(self.get_text_features(t_ids, t_mask), p=2, dim=-1)
        scale = self.logit_scale.exp().clamp(max=100)
        logits = scale * pf @ tf.t()
        return logits, logits.t()

clip_model = ProteinTextCLIP(model, text_model, PROT_DIM, TEXT_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(clip_model.parameters(), lr=LR)

class ProteinTextDataset(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        s = " ".join(list(re.sub(r"[UZOB]", "X", r["sequence"].upper())))
        return s, r["description"]

def clip_collate(batch):
    s, t = zip(*batch)
    pi = tokenizer(list(s), padding=True, truncation=True, max_length=TRUNCATE_SEQ_LEN, return_tensors="pt").to(DEVICE)
    ti = text_tokenizer(list(t), padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
    return pi, ti

loader = DataLoader(ProteinTextDataset(df_train_clean), batch_size=BATCH_SIZE, shuffle=True, collate_fn=clip_collate)
clip_model.train()
print("Training CLIP ...")
for step, (pb, tb) in enumerate(tqdm(loader)):
    lp, lt = clip_model(pb.input_ids, pb.attention_mask, tb.input_ids, tb.attention_mask)
    labels = torch.arange(lp.size(0)).to(DEVICE)
    loss = (F.cross_entropy(lp, labels) + F.cross_entropy(lt, labels)) / 2
    if torch.isnan(loss): continue
    loss.backward(); optimizer.step(); optimizer.zero_grad()
print("Done.")

## 12 · Protein-Text Retrieval (Testing Data)

Evaluation on the test set.

In [ ]:
clip_model.eval()
test_loader = DataLoader(ProteinTextDataset(df_test_clean), batch_size=BATCH_SIZE, shuffle=False, collate_fn=clip_collate)
p_feats, t_feats = [], []
with torch.no_grad():
    for pb, tb in tqdm(test_loader):
        pf = clip_model.get_prot_features(pb.input_ids, pb.attention_mask)
        tf = clip_model.get_text_features(tb.input_ids, tb.attention_mask)
        p_feats.append(F.normalize(pf, p=2, dim=-1))
        t_feats.append(F.normalize(tf, p=2, dim=-1))
p_feats, t_feats = torch.cat(p_feats), torch.cat(t_feats)
sim_matrix = p_feats @ t_feats.t()

def get_metrics(m):
    t = torch.arange(m.size(0)).to(m.device)
    p2t = m.topk(5, dim=1)[1]
    t2p = m.t().topk(5, dim=1)[1]
    return {"P2T@R1": (p2t[:, 0]==t).float().mean().item(), "T2P@R1": (t2p[:, 0]==t).float().mean().item()}

m = get_metrics(sim_matrix)
print(f"Retrieval: P2T R@1: {m['P2T@R1']*100:.1f}%, T2P R@1: {m['T2P@R1']*100:.1f}%")

print("\n--- Examples ---")
for i in range(2):
    idx = sim_matrix[i].argmax().item()
    print(f"Prot {i} Truth: {df_test_clean['description'].iloc[i][:100]}...")
    print(f"Prot {i} Pred : {df_test_clean['description'].iloc[idx][:100]}... {'✅' if i==idx else '❌'}")

## 13 · Custom Text-to-Protein Inference

Search for the most relevant protein given an arbitrary text description.

In [28]:
def retrieve_proteins(query_text, top_k=3):
    clip_model.eval()
    with torch.no_grad():
        # Encode query text
        t_in = text_tokenizer([query_text], padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        query_feat = clip_model.get_text_features(t_in.input_ids, t_in.attention_mask)
        query_feat = F.normalize(query_feat, p=2, dim=-1)
        
        # Calculate similarity with all test proteins
        cos_sim = (query_feat @ p_feats.t()).squeeze(0)
        top_vals, top_indices = cos_sim.topk(top_k)
        
        print(f"\nQuery: '{query_text}'")
        print(f"Top {top_k} Matches in Test Set:")
        for rank, (idx, val) in enumerate(zip(top_indices, top_vals)):
            idx = idx.item()
            print(f"{rank+1}. Score: {val:.4f}")
            print(f"   Sequence    : {df_test_clean['sequence'].iloc[idx][:100]}...")
            print(f"   Description : {df_test_clean['description'].iloc[idx]}")
            print("-" * 20)

# ── Example Usage ──
sample_query = "A protein involved in the transport of ions across the cell membrane."
retrieve_proteins(sample_query)


Query: 'A protein involved in the transport of ions across the cell membrane.'
Top 3 Matches in Test Set:
1. Score: nan
   Sequence    : MARGAALALLLFGLLGVLVAAPDGGFDLSDALPDNENKKPTAIPKKPSAGDDFDLGDAVVDGENDDPRPPNPPKPMPNPNPNHPSSSGSFSDADLADGVS...
   Description : No, it is the beta-3 adrenergic receptor that is expressed at a higher level in cells from metastatic or relapsed patients.
--------------------
2. Score: nan
   Sequence    : MWRAGSMSAELGVGCALRAVNERVQQAVARRPRDLPAIQPRLVAVSKTKPADMVIEAYGHGQRTFGENYVQELLEKASNPKILSLCPEIKWHFIGHLQKQ...
   Description : They lack EEG correlation and may be associated with a suspicious movement disorder, rarely reported in similar disorders.
--------------------
3. Score: nan
   Sequence    : MANSGLQLLGFSMALLGWVGLVACTAIPQWQMSSYAGDNIITAQAMYKGLWMDCVTQSTGMMSCKMYDSVLALSAALQATRALMVVSLVLGFLAMFVATM...
   Description : Yes, the protein is likely involved in regulating the movement of substances through cell-cell connections.
--------------------


## 14 · Visualize Top Retrievals in 3D

Predict structure and visualize the top 3 sequences using the ESMFold API (lightweight folding directly in the notebook).

In [29]:
!pip install -q py3Dmol requests
import py3Dmol
import requests
from IPython.display import display, HTML

def fold_and_visualize(sequence, rank, score):
    """Predicts the 3D structure using the ESMFold public API and displays it using py3Dmol."""
    print(f"\n{'='*50}\nMatch {rank} | Score: {score:.4f}\n{'='*50}")
    print(f"Sequence Length: {len(sequence)} aa")
    print("Querying ESMFold API for structure prediction...")
    
    # ESMFold REST API endpoint
    url = 'https://api.esmatlas.com/foldSequence/v1/pdb/'
    headers = {'Content-Type': 'text/plain'}
    
    try:
        response = requests.post(url, data=sequence, headers=headers)
        response.raise_for_status() # Raise exception for bad status codes
        pdb_string = response.text
        
        print("Structure predicted successfully! Visualizing...")
        # Visualize using py3Dmol inline
        view = py3Dmol.view(width=800, height=400)
        view.addModel(pdb_string, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.setBackgroundColor('white')
        view.zoomTo()
        view.show()
        
    except requests.exceptions.RequestException as e:
        print(f"\nError predicting structure: {e}")
        print("This might happen if the sequence is too long for the free API or the server is busy.")


def fetch_and_visualize_top_k(query_text, top_k=3):
    """Runs the retrieval logic and folds the top K results."""
    clip_model.eval()
    with torch.no_grad():
        # 1. Encode text query
        t_in = text_tokenizer([query_text], padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        query_feat = clip_model.get_text_features(t_in.input_ids, t_in.attention_mask)
        query_feat = F.normalize(query_feat, p=2, dim=-1)
        
        # 2. Get Cosine Similarities against all test proteins
        cos_sim = (query_feat @ p_feats.t()).squeeze(0)
        top_vals, top_indices = cos_sim.topk(top_k)
        
        print(f"\nSearching for: '{query_text}'\n")
        
        # 3. Fold and visualize each
        for rank, (idx, val) in enumerate(zip(top_indices, top_vals)):
            idx = idx.item()
            sequence = df_test_clean['sequence'].iloc[idx]
            fold_and_visualize(sequence, rank=rank+1, score=val.item())

# ── Example Usage ── 
# Make sure to run the cell above to define sample_query first!
fetch_and_visualize_top_k(sample_query, top_k=3)


Searching for: 'A protein involved in the transport of ions across the cell membrane.'


Match 1 | Score: nan
Sequence Length: 185 aa
Querying ESMFold API for structure prediction...
Structure predicted successfully! Visualizing...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Match 2 | Score: nan
Sequence Length: 256 aa
Querying ESMFold API for structure prediction...
Structure predicted successfully! Visualizing...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Match 3 | Score: nan
Sequence Length: 211 aa
Querying ESMFold API for structure prediction...
Structure predicted successfully! Visualizing...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Stage 4 — Lightweight Protein Sequence Decoder

ProtT5 is a full T5 encoder-decoder, but its decoder **cross-attends to the full encoder hidden-state sequence**, not a single mean-pooled vector. It was also pre-trained with span corruption, not full-sequence reconstruction. So we train a small **GRU decoder** (~2 M params) that maps a 1024-d ProtT5 embedding back to an AA sequence.

Pipeline: `AA sequence → ProtT5 encoder → 1024-d embedding → GRU decoder → AA sequence`

In [38]:
# ── AA Vocabulary ─────────────────────────────────────────────────────────────
AA_TOKENS = ['<pad>', '<sos>', '<eos>',
             'A','C','D','E','F','G','H','I','K','L',
             'M','N','P','Q','R','S','T','V','W','Y','X']

aa_to_idx = {aa: i for i, aa in enumerate(AA_TOKENS)}
idx_to_aa = {i: aa for i, aa in enumerate(AA_TOKENS)}
AA_VOCAB_SIZE = len(AA_TOKENS)
PAD_IDX = aa_to_idx['<pad>']
SOS_IDX = aa_to_idx['<sos>']
EOS_IDX = aa_to_idx['<eos>']

print(f"AA vocab size: {AA_VOCAB_SIZE}  (20 standard AAs + pad/sos/eos/X)")


# ── Lightweight GRU Decoder ──────────────────────────────────────────────────
class ProteinSeqDecoder(nn.Module):
    """GRU decoder: 1024-d protein embedding → AA token sequence."""
    def __init__(self, embed_dim=1024, hidden_dim=512, vocab_size=AA_VOCAB_SIZE,
                 num_layers=2, dropout=0.1, max_len=TRUNCATE_SEQ_LEN + 2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.max_len = max_len

        self.latent_to_hidden = nn.Linear(embed_dim, hidden_dim * num_layers)
        self.token_embed = nn.Embedding(vocab_size, hidden_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.out_proj = nn.Linear(hidden_dim, vocab_size)

    def _init_hidden(self, protein_emb):
        """Project protein embedding → GRU initial hidden state."""
        B = protein_emb.size(0)
        h = self.latent_to_hidden(protein_emb)            # [B, hidden*layers]
        h = h.view(B, self.num_layers, self.hidden_dim)
        return h.permute(1, 0, 2).contiguous()             # [layers, B, hidden]

    def forward(self, protein_emb, target_seq=None, teacher_forcing_ratio=0.5):
        """
        protein_emb : [B, embed_dim]
        target_seq  : [B, seq_len]  token indices (<sos> ... <eos> <pad>)
        Returns     : logits [B, T-1, vocab_size]
        """
        B = protein_emb.size(0)
        hidden = self._init_hidden(protein_emb)
        max_t = target_seq.size(1) if target_seq is not None else self.max_len

        inp = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=protein_emb.device)
        logits_list = []

        for t in range(max_t - 1):
            emb = self.token_embed(inp)             # [B, 1, hidden]
            out, hidden = self.gru(emb, hidden)     # out [B, 1, hidden]
            logit = self.out_proj(out)               # [B, 1, vocab]
            logits_list.append(logit)

            if target_seq is not None and random.random() < teacher_forcing_ratio:
                inp = target_seq[:, t + 1 : t + 2]
            else:
                inp = logit.argmax(dim=-1)

        return torch.cat(logits_list, dim=1)         # [B, T-1, vocab]

    @torch.no_grad()
    def generate(self, protein_emb, max_len=None):
        """Greedy-decode sequences from protein embeddings."""
        self.eval()
        logits = self.forward(protein_emb, target_seq=None, teacher_forcing_ratio=0.0)
        tokens = logits.argmax(dim=-1)               # [B, T]
        seqs = []
        for i in range(tokens.size(0)):
            chars = []
            for t in tokens[i]:
                aa = idx_to_aa[t.item()]
                if aa == '<eos>':
                    break
                if aa not in ('<pad>', '<sos>'):
                    chars.append(aa)
            seqs.append(''.join(chars))
        return seqs


decoder = ProteinSeqDecoder(embed_dim=1024, hidden_dim=512).to(DEVICE)
n_params = sum(p.numel() for p in decoder.parameters())
print(f"Decoder parameters: {n_params:,}  ({n_params/1e6:.1f} M)")

AA vocab size: 24  (20 standard AAs + pad/sos/eos/X)
Decoder parameters: 4,226,072  (4.2 M)


# Stage 4a — Train GRU Decoder (AA Sequence Reconstruction)

Before training the diffusion model, we train the GRU decoder to reconstruct AA sequences from ProtT5 embeddings. This gives us a working **embedding → sequence** path needed for decoding diffusion samples.

Pipeline: `AA sequence → ProtT5 encoder → 1024-d embedding → GRU decoder → AA sequence`


In [39]:
def seq_to_tensor(seq: str, max_len: int = TRUNCATE_SEQ_LEN + 2):
    """Convert an AA string → padded token tensor  [<sos> ... <eos> <pad>...]."""
    seq = re.sub(r"[UZOB]", "X", seq.upper())[:max_len - 2]
    ids = [SOS_IDX] + [aa_to_idx.get(c, aa_to_idx["X"]) for c in seq] + [EOS_IDX]
    ids += [PAD_IDX] * (max_len - len(ids))
    return torch.tensor(ids[:max_len], dtype=torch.long)


class ProteinReconDataset(Dataset):
    def __init__(self, df):
        self.sequences = df["sequence"].tolist()

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        raw_seq = self.sequences[idx]
        # Build formatted input for ProtT5
        fmt = " ".join(list(re.sub(r"[UZOB]", "X", raw_seq.upper())))
        target = seq_to_tensor(raw_seq)
        return fmt, target


def recon_collate(batch):
    fmt_seqs, targets = zip(*batch)
    enc_in = tokenizer(
        list(fmt_seqs), padding=True, truncation=True,
        max_length=TRUNCATE_SEQ_LEN, return_tensors="pt"
    ).to(DEVICE)
    targets = torch.stack(targets).to(DEVICE)
    return enc_in, targets


# ── Encoder helper: extract mean-pooled ProtT5 embedding ────────────────────
@torch.no_grad()
def encode_batch(enc_in):
    base = model
    if hasattr(base, "encoder"):
        out = base.encoder(
            input_ids=enc_in.input_ids,
            attention_mask=enc_in.attention_mask
        )
    else:
        out = base(
            input_ids=enc_in.input_ids,
            attention_mask=enc_in.attention_mask
        )
    hs = out.last_hidden_state          # [B, L, 1024]
    mask = enc_in.attention_mask.unsqueeze(-1).float()
    emb = (hs * mask).sum(1) / mask.sum(1)  # mean-pool over valid tokens
    return emb.float()


DECODER_EPOCHS    = 100
DECODER_BATCH     = 4
DECODER_LR        = 1e-3
TEACHER_FORCING   = 0.5

recon_loader = DataLoader(
    ProteinReconDataset(df_train_clean),
    batch_size=DECODER_BATCH, shuffle=True, collate_fn=recon_collate
)
dec_optimizer = torch.optim.AdamW(decoder.parameters(), lr=DECODER_LR)
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

decoder.train()
model.eval()   # keep ProtT5 frozen

print("Training GRU Decoder (1 epoch) …")
dec_total_loss = 0.0
for step, (enc_in, targets) in enumerate(tqdm(recon_loader, desc="Decoder")):
    emb = encode_batch(enc_in)                      # [B, 1024]
    logits = decoder(emb, target_seq=targets,       # [B, T-1, vocab]
                     teacher_forcing_ratio=TEACHER_FORCING)
    # targets: <sos> t1 t2 … <eos>  →  predict t1 … <eos>
    loss = ce_loss_fn(
        logits.reshape(-1, AA_VOCAB_SIZE),
        targets[:, 1:].reshape(-1)
    )
    if not torch.isnan(loss):
        loss.backward()
        dec_optimizer.step()
        dec_optimizer.zero_grad()
        dec_total_loss += loss.item()

avg_dec_loss = dec_total_loss / len(recon_loader)
print(f"GRU Decoder training done. Avg loss: {avg_dec_loss:.4f}")

# Quick sanity-check: reconstruct a training sequence
decoder.eval()
with torch.no_grad():
    sample_fmt = " ".join(list(re.sub(r"[UZOB]", "X", df_train_clean["sequence"].iloc[0].upper())))
    s_enc = tokenizer([sample_fmt], padding=True, truncation=True,
                       max_length=TRUNCATE_SEQ_LEN, return_tensors="pt").to(DEVICE)
    s_emb = encode_batch(s_enc)
    recon = decoder.generate(s_emb)[0]
orig = df_train_clean["sequence"].iloc[0][:50]
print(f"Original  (first 50 aa): {orig}")
print(f"Reconstructed (first 50 aa): {recon[:50]}")


Training GRU Decoder (1 epoch) …


Decoder:   0%|          | 0/25 [00:00<?, ?it/s]

GRU Decoder training done. Avg loss: 2.9814
Original  (first 50 aa): XWRLPRALCVHAAKTSKLSGPWSRPAAFMSTLLINQPQYAWLKELGLREE
Reconstructed (first 50 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ


# Stage 4b — Conditional Latent DDPM

We train a small **conditional DDPM** over the 1 024-d ProtT5 embedding space.  
The conditioning signal is the MiniLM text embedding (384-d) projected to 1 024-d.

Architecture:
- Noisy protein latent `zₜ` + text condition `c` → denoising MLP  
- Standard DDPM noise schedule (T = 1 000 linear betas)  
- Loss: simple MSE between predicted and actual noise (ε-prediction)


In [40]:
import math

# ── DDPM Noise Schedule ───────────────────────────────────────────────────────
DDPM_T       = 1000
BETA_START   = 1e-4
BETA_END     = 0.02
LATENT_DIM   = 1024
COND_DIM     = 384    # MiniLM output dimension

betas    = torch.linspace(BETA_START, BETA_END, DDPM_T).to(DEVICE)           # [T]
alphas   = 1.0 - betas                                                        # [T]
alpha_bar = torch.cumprod(alphas, dim=0)                                      # [T]
# Pre-compute for fast q(x_t | x_0)
sqrt_alpha_bar     = alpha_bar.sqrt()                                         # [T]
sqrt_one_minus_abar = (1.0 - alpha_bar).sqrt()                               # [T]


def q_sample(x0, t, noise=None):
    """
    Forward diffusion: sample x_t given x_0 and timestep t.
    x0  : [B, latent_dim]
    t   : [B]  (int indices into the schedule)
    """
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab   = sqrt_alpha_bar[t].unsqueeze(1)          # [B, 1]
    sqrt_1mab = sqrt_one_minus_abar[t].unsqueeze(1)     # [B, 1]
    return sqrt_ab * x0 + sqrt_1mab * noise, noise


# ── Sinusoidal Timestep Embedding ────────────────────────────────────────────
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)   # [B, dim]


# ── Conditional Denoising MLP ────────────────────────────────────────────────
class ConditionalDenoiseMLP(nn.Module):
    """
    Inputs : z_t (noisy latent), t (timestep), c (text condition)
    Output : predicted noise ε  (same shape as z_t)
    """
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM,
                 time_emb_dim=256, hidden_dim=1024):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbed(time_emb_dim)
        self.cond_proj = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.SiLU(),
        )
        in_dim = latent_dim + time_emb_dim + hidden_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, z_t, t, c):
        """
        z_t : [B, latent_dim]
        t   : [B]
        c   : [B, cond_dim]  — text embedding
        """
        t_emb = self.time_emb(t)           # [B, time_emb_dim]
        c_emb = self.cond_proj(c)          # [B, hidden_dim]
        inp   = torch.cat([z_t, t_emb, c_emb], dim=-1)
        return self.net(inp)               # [B, latent_dim]


denoiser = ConditionalDenoiseMLP().to(DEVICE)
n_ddpm = sum(p.numel() for p in denoiser.parameters())
print(f"DDPM denoiser parameters: {n_ddpm:,}  ({n_ddpm/1e6:.1f} M)")


DDPM denoiser parameters: 5,903,360  (5.9 M)


## 15 · Pre-compute Protein & Text Embeddings for the Training Set

We cache ProtT5 and MiniLM embeddings for every training pair so that the DDPM dataloader does not pay the encoding cost on every step.


In [41]:
EMBED_BATCH = 16   # smaller batch to avoid OOM on CPU

model.eval()
text_model.eval()

all_prot_embs = []
all_text_embs = []

print("Pre-computing training embeddings …")
for start in tqdm(range(0, len(df_train_clean), EMBED_BATCH), desc="Encoding"):
    batch = df_train_clean.iloc[start : start + EMBED_BATCH]

    # ── Protein embeddings via ProtT5 ─────────────────────────────────────
    seqs_fmt = [
        " ".join(list(re.sub(r"[UZOB]", "X", s.upper())))
        for s in batch["sequence"].tolist()
    ]
    p_enc = tokenizer(
        seqs_fmt, padding=True, truncation=True,
        max_length=TRUNCATE_SEQ_LEN, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        p_emb = encode_batch(p_enc).cpu()      # [B, 1024]

    # ── Text embeddings via MiniLM ────────────────────────────────────────
    descs = batch["description"].tolist()
    t_enc = text_tokenizer(
        descs, padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        t_hs  = text_model(**t_enc).last_hidden_state   # [B, L, 384]
        t_mask = t_enc.attention_mask.unsqueeze(-1).float()
        t_emb = (t_hs * t_mask).sum(1) / t_mask.sum(1) # mean-pool
        t_emb = t_emb.cpu()                             # [B, 384]

    all_prot_embs.append(p_emb)
    all_text_embs.append(t_emb)

train_prot_embs = torch.cat(all_prot_embs, dim=0)   # [N_TRAIN, 1024]
train_text_embs = torch.cat(all_text_embs, dim=0)   # [N_TRAIN, 384]

print(f"Protein embeddings : {train_prot_embs.shape}")
print(f"Text    embeddings : {train_text_embs.shape}")


Pre-computing training embeddings …


Encoding:   0%|          | 0/7 [00:00<?, ?it/s]

Protein embeddings : torch.Size([100, 1024])
Text    embeddings : torch.Size([100, 384])


## 16 · Train Conditional Latent DDPM (1 Epoch)

For each training step we:
1. Sample a random timestep `t ~ Uniform(1, T)`.
2. Add noise via the forward process `q(z_t | z_0)`.
3. Ask the denoiser to predict the noise `ε`.
4. Minimise MSE(ε_pred, ε_true).


In [42]:
from torch.utils.data import TensorDataset

DDPM_BATCH = 16
DDPM_LR    = 2e-4

# ── Normalise protein embeddings (zero-mean unit-var per dimension) ──────────
prot_mean = train_prot_embs.mean(0)
prot_std  = train_prot_embs.std(0).clamp(min=1e-6)
train_prot_norm = (train_prot_embs - prot_mean) / prot_std

ddpm_dataset = TensorDataset(train_prot_norm, train_text_embs)
ddpm_loader  = DataLoader(ddpm_dataset, batch_size=DDPM_BATCH, shuffle=True, drop_last=True)

ddpm_optimizer = torch.optim.AdamW(denoiser.parameters(), lr=DDPM_LR)
mse_loss = nn.MSELoss()

denoiser.train()
print("Training Conditional Latent DDPM (1 epoch) …")

ddpm_total_loss = 0.0
for step, (z0_batch, c_batch) in enumerate(tqdm(ddpm_loader, desc="DDPM")):
    z0_batch = z0_batch.to(DEVICE)   # [B, 1024]
    c_batch  = c_batch.to(DEVICE)    # [B, 384]

    # ── Sample random timesteps ───────────────────────────────────────────
    t = torch.randint(0, DDPM_T, (z0_batch.size(0),), device=DEVICE)

    # ── Forward diffusion ─────────────────────────────────────────────────
    z_t, noise = q_sample(z0_batch, t)

    # ── Predict noise ─────────────────────────────────────────────────────
    noise_pred = denoiser(z_t, t, c_batch)

    loss = mse_loss(noise_pred, noise)
    loss.backward()
    ddpm_optimizer.step()
    ddpm_optimizer.zero_grad()
    ddpm_total_loss += loss.item()

avg_ddpm_loss = ddpm_total_loss / len(ddpm_loader)
print(f"DDPM training done. Avg MSE loss: {avg_ddpm_loss:.6f}")


Training Conditional Latent DDPM (1 epoch) …


DDPM:   0%|          | 0/6 [00:00<?, ?it/s]

DDPM training done. Avg MSE loss: 0.997639


## 17 · Prompt-to-Protein: Sample from DDPM → Decode AA Sequence

Given a free-text prompt we:
1. Encode the prompt with MiniLM → conditioning vector `c`.
2. Start from pure Gaussian noise `z_T`.
3. Iteratively denoise using the DDPM reverse process (DDPM ancestral sampling).
4. Un-normalise the resulting latent and decode an AA sequence with the GRU decoder.


In [43]:
@torch.no_grad()
def ddpm_reverse_sample(c: torch.Tensor) -> torch.Tensor:
    """
    DDPM ancestral sampling (full T-step reverse process).
    c : [B, 384] text conditioning embeddings
    Returns z_0_hat : [B, 1024] (normalised latent space)
    """
    denoiser.eval()
    B = c.size(0)
    z = torch.randn(B, LATENT_DIM, device=DEVICE)   # start from z_T ~ N(0,I)

    for t_idx in tqdm(reversed(range(DDPM_T)), desc="DDPM reverse", total=DDPM_T, leave=False):
        t_tensor = torch.full((B,), t_idx, device=DEVICE, dtype=torch.long)

        beta_t    = betas[t_idx]
        alpha_t   = alphas[t_idx]
        abar_t    = alpha_bar[t_idx]
        abar_prev = alpha_bar[t_idx - 1] if t_idx > 0 else torch.tensor(1.0, device=DEVICE)

        noise_pred = denoiser(z, t_tensor, c)

        # Reconstruct x_0 estimate
        x0_hat = (z - (1 - abar_t).sqrt() * noise_pred) / abar_t.sqrt()
        x0_hat = x0_hat.clamp(-5.0, 5.0)   # soft clamp for stability

        # Compute posterior mean
        coef1 = (abar_prev.sqrt() * beta_t)              / (1.0 - abar_t)
        coef2 = (alpha_t.sqrt()   * (1.0 - abar_prev))  / (1.0 - abar_t)
        mu    = coef1 * x0_hat + coef2 * z

        if t_idx > 0:
            posterior_var = beta_t * (1.0 - abar_prev) / (1.0 - abar_t)
            z = mu + posterior_var.sqrt() * torch.randn_like(z)
        else:
            z = mu

    return z   # z_0_hat in normalised space


@torch.no_grad()
def prompt_to_protein(prompt: str, top_k_retrieval: int = 3):
    """
    End-to-end: text prompt -> AA sequence(s).

    1. Encode prompt with MiniLM.
    2. Sample z_0 from the DDPM conditioned on the text.
    3. Un-normalise z_0.
    4a. Decode directly with the GRU decoder.
    4b. Retrieve nearest neighbour sequences from the training embeddings.
    """
    # 1. Text condition
    t_enc = text_tokenizer(
        [prompt], padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    ).to(DEVICE)
    t_hs   = text_model(**t_enc).last_hidden_state
    t_mask = t_enc.attention_mask.unsqueeze(-1).float()
    c      = (t_hs * t_mask).sum(1) / t_mask.sum(1)    # [1, 384]

    # 2. DDPM sample
    z_norm = ddpm_reverse_sample(c)                     # [1, 1024]

    # 3. Un-normalise
    z_hat = z_norm * prot_std.to(DEVICE) + prot_mean.to(DEVICE)

    # 4a. GRU decode
    decoder.eval()
    decoded_seqs = decoder.generate(z_hat)              # list[str]

    # 4b. Nearest-neighbour retrieval from the training set
    cos_sims = F.cosine_similarity(
        z_norm.cpu().expand(len(train_prot_norm), -1),
        train_prot_norm, dim=-1
    )
    top_ids   = cos_sims.topk(top_k_retrieval).indices.tolist()
    retrieved = df_train_clean["sequence"].iloc[top_ids].tolist()
    ret_descs = df_train_clean["description"].iloc[top_ids].tolist()

    return decoded_seqs, retrieved, ret_descs


# Run inference on example prompts
prompts = [
    "a metal transporter protein",
    "a kinase-like signaling protein",
    "a small enzyme involved in redox chemistry",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt : {prompt}")
    decoded, retrieved, ret_descs = prompt_to_protein(prompt, top_k_retrieval=2)
    print(f"GRU-decoded sequence  (first 80 aa): {decoded[0][:80]}{'...' if len(decoded[0]) > 80 else ''}")
    print(f"Nearest retrieved seq (first 80 aa): {retrieved[0][:80]}{'...' if len(retrieved[0]) > 80 else ''}")
    print(f"Nearest retrieved description      : {ret_descs[0][:120]}")



Prompt : a metal transporter protein


DDPM reverse:   0%|          | 0/1000 [00:00<?, ?it/s]

GRU-decoded sequence  (first 80 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ...
Nearest retrieved seq (first 80 aa): MKIPNIGNVMNKFEILGVVGEGAYGVVLKCRHKETHEIVAIKKFKDSEENEEVKETTLRELKMLRTLKQENIVELKEAFR...
Nearest retrieved description      : Yes, individuals with a deficiency of the protein often have associated behavioral abnormalities, such as epilepsy and a

Prompt : a kinase-like signaling protein


DDPM reverse:   0%|          | 0/1000 [00:00<?, ?it/s]

GRU-decoded sequence  (first 80 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ...
Nearest retrieved seq (first 80 aa): MGRVGEIPPPPPEDFPLPPPPLAGDGDDAEGALGGAFPPPPPPIEESFPPAPLEEEIFPSPPPPPEEEGGPEAPIPPPPQ...
Nearest retrieved description      : α-catenin.

Prompt : a small enzyme involved in redox chemistry


DDPM reverse:   0%|          | 0/1000 [00:00<?, ?it/s]

GRU-decoded sequence  (first 80 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ...
Nearest retrieved seq (first 80 aa): MPLDDLDREDEVRLLKYLFTLIRAGMTEEAQRLCKRCGQAWRAATLEGWKLYHDPNVNGGTELEPVEGNPYRRIWKISCW...
Nearest retrieved description      : The interaction of this protein is studied using DNA origami nanostructures and in a cellular environment.


## 18 · Animate Protein Structure Evolution During Denoising

We capture **8 latent snapshots** evenly spaced across the 1 000-step reverse diffusion (from pure noise `t=999` to the final sample `t=0`), decode each snapshot to an AA sequence with the GRU decoder, fold every sequence with the **ESMFold public API**, and display an interactive animated viewer with a **play button** and scrubber.

> **Note:** This cell makes up to 8 network requests to `api.esmatlas.com`. Allow ~10–30 s per sequence.


In [44]:
%pip install -q ipywidgets py3Dmol requests

import requests, math, time
import ipywidgets as widgets
from IPython.display import display, HTML
import py3Dmol

# ── 1. Reverse sampler that saves intermediate latents ───────────────────────
N_SNAPSHOTS = 8   # number of denoising frames to capture (including t≈999 and t=0)

@torch.no_grad()
def ddpm_sample_with_snapshots(c: torch.Tensor, n_snapshots: int = N_SNAPSHOTS):
    """
    Full DDPM reverse pass that records `n_snapshots` latents
    evenly distributed across the denoising trajectory.

    Returns
    -------
    z_final   : [1, 1024]  final clean latent
    snapshots : list[(t_idx: int, z: Tensor[1,1024])]  high-t → low-t order
    """
    denoiser.eval()
    B = c.size(0)
    z = torch.randn(B, LATENT_DIM, device=DEVICE)

    # choose which t_idx values to snapshot (evenly spaced, always include t=0)
    snap_at = set(
        int(round(x))
        for x in np.linspace(DDPM_T - 1, 0, n_snapshots)
    )
    snap_at.add(0)

    snapshots = []

    for t_idx in tqdm(reversed(range(DDPM_T)),
                      desc="Reverse diffusion", total=DDPM_T, leave=False):
        t_tensor  = torch.full((B,), t_idx, device=DEVICE, dtype=torch.long)
        beta_t    = betas[t_idx]
        alpha_t   = alphas[t_idx]
        abar_t    = alpha_bar[t_idx]
        abar_prev = alpha_bar[t_idx - 1] if t_idx > 0 else torch.tensor(1.0, device=DEVICE)

        noise_pred = denoiser(z, t_tensor, c)
        x0_hat = (z - (1 - abar_t).sqrt() * noise_pred) / abar_t.sqrt()
        x0_hat = x0_hat.clamp(-5.0, 5.0)

        coef1 = (abar_prev.sqrt() * beta_t)             / (1.0 - abar_t)
        coef2 = (alpha_t.sqrt()   * (1.0 - abar_prev)) / (1.0 - abar_t)
        mu    = coef1 * x0_hat + coef2 * z

        if t_idx > 0:
            posterior_var = beta_t * (1.0 - abar_prev) / (1.0 - abar_t)
            z = mu + posterior_var.sqrt() * torch.randn_like(z)
        else:
            z = mu

        if t_idx in snap_at:
            snapshots.append((t_idx, z.clone().cpu()))

    # sort: noisiest frame first (high t_idx → low t_idx)
    snapshots.sort(key=lambda x: x[0], reverse=True)
    return z, snapshots


# ── 2. Fold a sequence via ESMFold REST API ───────────────────────────────────
def fold_sequence(sequence: str, retries: int = 2) -> str | None:
    """Return a PDB string from ESMFold, or None on failure."""
    url     = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    headers = {"Content-Type": "text/plain"}
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, data=sequence, headers=headers, timeout=60)
            resp.raise_for_status()
            return resp.text
        except Exception as exc:
            if attempt < retries:
                time.sleep(3)
            else:
                print(f"  ✗ Folding failed after {retries+1} attempts: {exc}")
                return None


# ── 3. Run the pipeline for an animation prompt ───────────────────────────────
ANIM_PROMPT = "a metal transporter protein"   # ← change freely

print(f'Generating denoising snapshots for: "{ANIM_PROMPT}"\n')

# Encode text condition
with torch.no_grad():
    t_enc  = text_tokenizer([ANIM_PROMPT], padding=True, truncation=True,
                             max_length=128, return_tensors="pt").to(DEVICE)
    t_hs   = text_model(**t_enc).last_hidden_state
    t_mask = t_enc.attention_mask.unsqueeze(-1).float()
    cond   = (t_hs * t_mask).sum(1) / t_mask.sum(1)     # [1, 384]

# Reverse diffusion with snapshot capture
_, snapshots = ddpm_sample_with_snapshots(cond, n_snapshots=N_SNAPSHOTS)

# Decode each snapshot latent → AA sequence
decoder.eval()
frame_seqs   = []
frame_labels = []
for t_val, z_snap in snapshots:
    z_hat = z_snap.to(DEVICE) * prot_std.to(DEVICE) + prot_mean.to(DEVICE)
    seq   = decoder.generate(z_hat)[0]
    seq   = seq if len(seq) > 5 else "ACDEFGHIKLMNPQRSTVWY"   # guard empty decode
    frame_seqs.append(seq)
    noise_pct = round(100 * t_val / (DDPM_T - 1)) if t_val > 0 else 0
    frame_labels.append(f"t = {t_val:>4d}  |  noise ≈ {noise_pct:>3d}%")
    print(f"  [{t_val:>4d}] seq (first 60 aa): {seq[:60]}{'…' if len(seq) > 60 else ''}")

# Fold each sequence with ESMFold
print(f"\nFolding {len(frame_seqs)} sequences via ESMFold …")
pdb_strings = []
valid_labels = []
for i, (seq, lbl) in enumerate(zip(frame_seqs, frame_labels)):
    print(f"  [{i+1}/{len(frame_seqs)}] {lbl}  (len={len(seq)})")
    pdb = fold_sequence(seq)
    if pdb:
        pdb_strings.append(pdb)
        valid_labels.append(lbl)
    else:
        print(f"    Skipping frame — folding failed.")

print(f"\nSuccessfully folded {len(pdb_strings)} / {len(frame_seqs)} frames.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.2 MB/s eta 0:00:0000:01
Generating denoising snapshots for: "a metal transporter protein"



Reverse diffusion:   0%|          | 0/1000 [00:00<?, ?it/s]

  [ 999] seq (first 60 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 856] seq (first 60 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 714] seq (first 60 aa): MMLLLPLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 571] seq (first 60 aa): MMLLLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 428] seq (first 60 aa): MMLLLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 285] seq (first 60 aa): MMLLLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [ 143] seq (first 60 aa): MMLLLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…
  [   0] seq (first 60 aa): MMLLLPLPLQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ…

Folding 8 sequences via ESMFold …
  [1/8] t =  999  |  noise ≈ 100%  (len=257)
  [2/8] t =  856  |  noise ≈  86%  (len=257)
  [3/8] t =  714  |  noise ≈  71%  (len=257)
  [4/8] t =  571  |  noise ≈  57%  (len=257)
  [5/8] t =  428  |  noise ≈  43%  (len=257)
  [6/8] t =  285  | 

In [46]:
# ── 4. Interactive animated viewer ───────────────────────────────────────────
# Uses ipywidgets Play + IntSlider linked to a py3Dmol output widget.
# The Play button auto-advances frames (~1 s per frame).

if not pdb_strings:
    print("No folded structures available — cannot render animation.")
else:
    # Colour palette cycles through spectrum / chain / residue
    COLOR_SCHEMES = ["spectrum", "chainHetatm", "ssJmol"]

    # ── Widget layout ────────────────────────────────────────────────────────
    play_btn = widgets.Play(
        value=0, min=0, max=len(pdb_strings) - 1,
        step=1, interval=1200,          # ms between frames
        description="▶ Play", disabled=False
    )
    slider = widgets.IntSlider(
        value=0, min=0, max=len(pdb_strings) - 1,
        step=1, description="Frame:",
        layout=widgets.Layout(width="420px"),
        style={"description_width": "55px"},
    )
    color_dd = widgets.Dropdown(
        options=COLOR_SCHEMES, value="spectrum",
        description="Colour:", layout=widgets.Layout(width="210px"),
        style={"description_width": "55px"},
    )
    label_out = widgets.Label(value=valid_labels[0] if valid_labels else "")
    viewer_out = widgets.Output()

    # link play ↔ slider
    widgets.jslink((play_btn, "value"), (slider, "value"))

    def render_frame(frame_idx, color_scheme):
        with viewer_out:
            viewer_out.clear_output(wait=True)
            pdb = pdb_strings[frame_idx]
            view = py3Dmol.view(width=760, height=480)
            view.addModel(pdb, "pdb")
            view.setStyle({"cartoon": {"color": color_scheme}})
            # Draw semi-transparent surface for context
            view.addSurface(
                py3Dmol.VDW,
                {"opacity": 0.12, "color": "white"},
                {"hetflag": False}
            )
            view.setBackgroundColor("#1a1a2e")   # dark background pops colours
            view.zoomTo()
            view.spin(False)
            view.show()
            label_out.value = (
                f"Frame {frame_idx + 1} / {len(pdb_strings)}   —   {valid_labels[frame_idx]}"
            )

    # hook slider changes
    def on_slider_change(change):
        render_frame(change["new"], color_dd.value)

    def on_color_change(change):
        render_frame(slider.value, change["new"])

    slider.observe(on_slider_change, names="value")
    color_dd.observe(on_color_change, names="value")

    # ── Layout & initial render ──────────────────────────────────────────────
    controls_row = widgets.HBox(
        [play_btn, slider, color_dd],
        layout=widgets.Layout(margin="4px 0 6px 0")
    )
    ui = widgets.VBox(
        [
            widgets.HTML(
                f"<h3 style='font-family:sans-serif;margin:6px 0'>Protein Denoising Animation"
                f"<br><small style='color:#888'>Prompt: \"{ANIM_PROMPT}\"</small></h3>"
            ),
            controls_row,
            label_out,
            viewer_out,
        ]
    )
    display(ui)
    render_frame(0, "spectrum")   # show the noisiest frame first
